# Module 2 Exercise: MERGE Is the Primary Key

Spark and Delta Lake do **not** enforce `PRIMARY KEY` constraints. There are
no identity columns on Lakehouse tables, no `FOREIGN KEY` references, no
unique indexes. So how do you handle upserts, deduplication, and incremental
loads?

With **MERGE**. The same `MERGE` statement you have written in T-SQL since
SQL Server 2008, compiled by Spark into a single atomic Delta commit.

| Lakehouse pattern                             | SQL Server equivalent                                |
|-----------------------------------------------|------------------------------------------------------|
| `MERGE INTO target USING source ON ...`       | `MERGE INTO target USING source ON ...` (same!)      |
| `WHEN NOT MATCHED THEN INSERT *`              | `WHEN NOT MATCHED THEN INSERT *` (same!)             |
| Idempotent re-runs via natural key            | PK + `IF NOT EXISTS` staging table dance             |
| `DESCRIBE HISTORY` shows every MERGE commit   | `msdb.dbo.backupset` / trace file for ETL audit      |

**The convention is the primary key.** Pick a natural key (here, `id` as
CustomerId), MERGE on it, and let Delta handle atomicity.

> In SQL Server the key is enforced. In Delta the key is a discipline.


## 1. Preview Silver, prepare Gold

The `customers` table from `module_0_1_DataBuilder` is our Silver layer,
cleansed rows with a stable natural key. Gold is the curated downstream copy.
We seed it empty so the first MERGE inserts every row.


In [ ]:
%%sql
SELECT id, name, country, city
FROM   customers
LIMIT  5


In [ ]:
%%sql
CREATE OR REPLACE TABLE gold_customers AS
SELECT * FROM customers WHERE 1 = 0


## 2. Run the MERGE, Silver to Gold

Three clauses, one statement:

- `WHEN MATCHED AND <columns differ>` updates rows whose payload changed.
- `WHEN NOT MATCHED` inserts rows that do not exist in Gold yet.
- Matched rows whose payload is unchanged are skipped, no write, no commit entry.


In [ ]:
%%sql
MERGE INTO gold_customers AS t
USING customers          AS s
ON    t.id = s.id
WHEN MATCHED AND (
       t.name    <> s.name
    OR t.country <> s.country
    OR t.email   <> s.email
    OR t.city    <> s.city
    OR t.phone   <> s.phone
) THEN UPDATE SET
       t.name    = s.name,
       t.country = s.country,
       t.email   = s.email,
       t.city    = s.city,
       t.phone   = s.phone
WHEN NOT MATCHED THEN INSERT *


In [ ]:
%%sql
SELECT COUNT(*) AS row_count FROM gold_customers


## 3. Re-run to prove idempotence

Run the identical MERGE a second time. Because every row is matched *and*
unchanged, Delta writes nothing new. Gold's row count does not move, and the
Delta log only gains a lightweight metadata entry.


In [ ]:
%%sql
MERGE INTO gold_customers AS t
USING customers          AS s
ON    t.id = s.id
WHEN MATCHED AND (
       t.name    <> s.name
    OR t.country <> s.country
    OR t.email   <> s.email
    OR t.city    <> s.city
    OR t.phone   <> s.phone
) THEN UPDATE SET
       t.name    = s.name,
       t.country = s.country,
       t.email   = s.email,
       t.city    = s.city,
       t.phone   = s.phone
WHEN NOT MATCHED THEN INSERT *


In [ ]:
%%sql
SELECT COUNT(*) AS row_count FROM gold_customers


## 4. Inspect the Delta history

Each MERGE shows up as its own commit in the Delta transaction log, with
`numTargetRowsInserted`, `numTargetRowsUpdated`, and
`numTargetRowsCopied` in the `operationMetrics`. Compare this to SQL Server
trace files or `msdb.dbo.backupset`, same audit, free with the format.


In [ ]:
%%sql
DESCRIBE HISTORY gold_customers


## Key takeaway

MERGE turns idempotence from a coding discipline into a language feature.
The natural key is the contract, the engine does the bookkeeping, and your
T-SQL muscle memory from SQL Server 2008 carries over unchanged.
